# 03 — Órdenes: crear, consultar, editar, cancelar

- Ver las **órdenes abiertas**.
- **Crear** una orden.
- El **detalle de una `order_id`**.
- **Editar** una orden pendiente.
- **Cancelar** (una, o todas).

## ⚠ Este cuaderno coloca órdenes REALES

Bitunix no tiene testnet. Las celdas que colocan están puestas a propósito
como órdenes **LIMIT muy lejos del precio** para que no lleguen a ejecutarse
y se puedan cancelar — pero son órdenes de verdad, con dinero de verdad.

Lee y ajusta `CANTIDAD` antes de ejecutar. Con Binance en testnet
(`testnet=True` + claves de testnet) esto es inofensivo.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # el repo, sin instalarlo

from luarix_exchange.brokers.factory import AvailableBrokers, BrokerFactory
from luarix_exchange.brokers.credentials import BrokerCredentials, load_dotenv

load_dotenv()   # lee el .env de la raiz del repo

# Cambia esto para probar el otro broker. Todo lo demas del cuaderno
# funciona igual: esa es justamente la idea del contrato unico.
BROKER   = AvailableBrokers.BITUNIX     # o AvailableBrokers.BINANCE
MARKET   = "futures"                    # o "spot"
SYMBOL   = "BTCUSDT"

# Sin `credentials=`, la factory las lee del .env.
# En produccion el backend las pasa explicitas, una por usuario:
#     BrokerFactory.get_broker(BROKER, credentials=BrokerCredentials(
#         api_key=fila.api_key, api_secret=fila.api_secret))
broker = BrokerFactory.get_broker(
    BROKER,
    testnet=False,          # Bitunix lo ignora: no tiene testnet
    test_order=False,       # solo Binance: True valida sin ejecutar
    trading_mode="live",
    allow_live_trading=False,   # sigue bloqueando el place_order real
)
print(type(broker).__name__, "| futuros:", broker.supports_futures, "| OCO:", broker.supports_oco)

## Órdenes abiertas

Devuelve `OrderInfo`, normalizado: los dos brokers llaman distinto a los
mismos campos (`origQty` vs `qty`, `executedQty` vs `tradeQty`, y el lado es
`"BUY"` en Binance pero `1` en el spot de Bitunix). El diccionario original
sigue entero en `.raw`.

⚠ **El spot de Bitunix exige un símbolo**: su endpoint no tiene un "dame
todas las de la cuenta". Sin `symbol` levanta en vez de devolver `[]` — una
lista vacía se leería como "no tienes órdenes", que es justo la conclusión
equivocada.

In [ ]:
abiertas = broker.get_open_orders(market=MARKET)
print(len(abiertas), "ordenes abiertas")
for o in abiertas:
    print(f"  {o.order_id:22} {o.symbol:10} {o.side.value if o.side else '?':5} "
          f"{o.order_type:8} qty={o.quantity:<10} @ {o.price} -> {o.status.value}")

In [ ]:
# Filtrado por par (obligatorio en el spot de Bitunix).
broker.get_open_orders(market=MARKET, symbol=SYMBOL)

## Crear una orden

`OrderRequest` es la instrucción exacta que se manda. Lo importante:

- `market` decide a qué cuenta va. Sin él, todo iría a spot.
- `tp_price` / `sl_price` viajan **dentro** de la orden en Bitunix futuros
  (`tpPrice`/`slPrice`) — es atómico: la posición nunca llega a existir sin
  protección. Binance no lo admite, así que ahí se colocan acto seguido como
  `TAKE_PROFIT_MARKET` / `STOP_MARKET` con `closePosition=true`.
- `reduce_only=True` marca la orden como **cierre**. En modo cobertura
  (HEDGE) no es cosmético: una venta sin esa marca *no* reduce un largo
  abierto, abre un corto independiente que convive con él. Eso ya pasó de
  verdad — ver `docs/plans/bugs.md`.

In [ ]:
from luarix_exchange.models.orders import OrderRequest, OrderType
from luarix_exchange.models.orders import Side

CANTIDAD = 0.001   # <-- AJUSTA ESTO

precio_actual = broker.get_ticker(SYMBOL, market=MARKET)
precio_limite = round(precio_actual * 0.80, 2)   # 20% por debajo: no llenara

peticion = OrderRequest(
    symbol=SYMBOL,
    side=Side.BUY,
    quantity=CANTIDAD,
    order_type=OrderType.LIMIT,
    price=precio_limite,
    market=MARKET,
)
print(f"mercado en {precio_actual} -> limite en {precio_limite}")
peticion

In [ ]:
# ⚠ DESCOMENTA PARA COLOCAR DE VERDAD
# ack, fills = broker.place_order(peticion)
# print("id:", ack.exchange_order_id, "| estado:", ack.status)
# ORDER_ID = ack.exchange_order_id

## Detalle de una `order_id`

Distinto de `get_order_status`, que solo da el enum: acá vienen además la
cantidad ejecutada, el precio medio real y el TP/SL adjunto.

Responde también por órdenes **ya cerradas**, y eso es la parte importante.
Derivar el estado de la lista de pendientes *no vale*: una orden ausente de
ahí puede haber llenado **o** haber sido cancelada a mano, y confundirlas ya
marcó como llenadas tres órdenes canceladas, abriendo posiciones que nunca
existieron (`docs/plans/bugs.md`).

In [ ]:
# ORDER_ID = "pega aqui un id"
# info = broker.get_order(SYMBOL, ORDER_ID, market=MARKET)
# print(f"{info.symbol} {info.side.value} {info.order_type}")
# print(f"  estado    : {info.status.value}  (viva={info.is_open})")
# print(f"  cantidad  : {info.filled_quantity} de {info.quantity}  (queda {info.remaining_quantity})")
# print(f"  precio    : limite={info.price}  medio_real={info.average_price}")
# print(f"  proteccion: TP={info.tp_price}  SL={info.sl_price}")
# info

In [ ]:
# Solo el estado, si es lo unico que hace falta.
# broker.get_order_status(SYMBOL, ORDER_ID, market=MARKET)

## Editar una orden

Lo que no se pasa se conserva de la orden original.

⚠ **El id puede cambiar.** Solo hay enmienda nativa en *futuros* (Bitunix
`modify_order`, Binance `futures_modify_order`). En **spot** no existe tal
cosa: la única vía es cancelar y volver a colocar, lo que da un
`exchange_order_id` **nuevo** y deja un hueco sin protección entre las dos
llamadas. Relee el id del `OrderAck` que se devuelve; no lo des por estable.

In [ ]:
# nuevo_limite = round(precio_actual * 0.85, 2)
# ack = broker.update_order(SYMBOL, ORDER_ID, price=nuevo_limite, market=MARKET)
# print("id ahora:", ack.exchange_order_id, "(antes", ORDER_ID + ")")
# ORDER_ID = ack.exchange_order_id

In [ ]:
# Tambien la cantidad, o el stop.
# broker.update_order(SYMBOL, ORDER_ID, quantity=0.002, market=MARKET)

## Cancelar

In [ ]:
# broker.cancel_order(SYMBOL, ORDER_ID, market=MARKET)

In [ ]:
# Todas las pendientes de un par -- devuelve los ids que cancelo.
# broker.cancel_open_orders(market=MARKET, symbol=SYMBOL)

## Ciclo completo (colocar → consultar → editar → cancelar)

Todo junto, para verificar de una pasada. Sigue colocando de verdad.

In [ ]:
def ciclo_completo():
    precio = broker.get_ticker(SYMBOL, market=MARKET)
    limite = round(precio * 0.80, 2)

    ack, _ = broker.place_order(OrderRequest(
        symbol=SYMBOL, side=Side.BUY, quantity=CANTIDAD,
        order_type=OrderType.LIMIT, price=limite, market=MARKET,
    ))
    oid = ack.exchange_order_id
    print("1. colocada     ->", oid, ack.status.value)

    info = broker.get_order(SYMBOL, oid, market=MARKET)
    print("2. consultada   ->", info.status.value, "qty", info.quantity, "@", info.price)

    ack2 = broker.update_order(SYMBOL, oid, price=round(precio * 0.85, 2), market=MARKET)
    oid = ack2.exchange_order_id
    print("3. editada      ->", oid)

    vivas = [o.order_id for o in broker.get_open_orders(market=MARKET, symbol=SYMBOL)]
    print("4. abiertas     ->", vivas)

    broker.cancel_order(SYMBOL, oid, market=MARKET)
    print("5. cancelada    ->", broker.get_order(SYMBOL, oid, market=MARKET).status.value)

# ⚠ DESCOMENTA PARA EJECUTARLO DE VERDAD
# ciclo_completo()

## Orden con TP y SL adjuntos (Bitunix futuros)

La forma nativa y la única atómica: la posición nunca llega a existir sin
stop. Binance acepta los mismos campos pero por dentro los resuelve con dos
órdenes aparte.

In [ ]:
# protegida = OrderRequest(
#     symbol=SYMBOL, side=Side.BUY, quantity=CANTIDAD,
#     order_type=OrderType.LIMIT, price=round(precio_actual * 0.80, 2),
#     market="futures",
#     tp_price=round(precio_actual * 0.90, 2),   # objetivo
#     sl_price=round(precio_actual * 0.75, 2),   # stop
# )
# ack, _ = broker.place_order(protegida)
# broker.get_order(SYMBOL, ack.exchange_order_id, market="futures")